# Uncertainty Quantification: Two-Level Bootstrap, Leak-Safe K-Fold CV, Test CI

Three independent pieces, all answering the same underlying question, how much confidence do the results already reported in `04_fusion_model.ipynb` and `05_ablation_study.ipynb` actually deserve, from three different angles:

- **Part A**: two-level bootstrap on all 7 ablation configs (val only) -- ensemble-level (resampling trajectories) and seed-level (resampling seeds) are kept explicitly separate, since they quantify different sources of uncertainty, and the paired delta significance test is run under both views so any place where the two disagree, on significance or on ranking direction, is surfaced with numbers rather than left as an impression from a table.
- **Part B**: a leak-safe 6-fold CV robustness check on Full Model, trained fresh per fold on a pool restricted to train+val rows only, with a programmatic assertion (not a comment) that no test-split row reaches the fold-generation pool.
- **Part C**: a post-hoc bootstrap confidence interval around the already-final test F1 from `04`. This does not touch test again in the sense that matters for this project's single-test-touch discipline -- the evaluation already happened exactly once; this only quantifies uncertainty around that fixed, unchangeable number.

This notebook makes no config-selection or architecture decision anywhere. It exists entirely to quantify how much confidence the existing results deserve.

## 0. Setup

In [ ]:
import csv
import random
import re
import json
import time
import math
import traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

MANIFEST = Path('~/Desktop/convscript/Code/Manifest/manifests/master_manifest.csv').expanduser()
ROOT     = Path('~/Desktop/Thesis/TR-6').expanduser()
GAS_NORM = Path('~/Desktop/convscript/Code/Manifest/GasNorm/gas_norm_stats.json').expanduser()
RUNS     = Path('~/Desktop/convscript/runs').expanduser()

DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f'Device: {DEVICE}')

FUSION_OUT   = RUNS / "04_fusion_model"
ABLATION_OUT = RUNS / "05_ablation_study"
OUT_DIR      = RUNS / "11_uncertainty_quantification"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_KEY = "model_a_winner"  # 04's own checkpoint save key
FAILURES = []
THRESHOLDS_TO_SWEEP = [t / 100 for t in range(10, 91)]


## Data pipeline -- row-filterable, split-aware by construction

`TrimodalDataset` and `DayLevelSequenceDataset` both take an optional `row_filter` callable, applied at the raw manifest-row level, before day-level aggregation. This matters specifically because of the leak risk in `13_kfold_cv.ipynb`: that notebook built its pool with `split=None`, which does not merely admit whole test trajectories into the pool, it merges individual train/val/test **days** into the same trajectory object, since split filtering happens on rows before trajectories are ever assembled. `row_filter` lets Part B restrict to `split in {train, val}` at the row level, correctly, before any aggregation happens -- not as a post-hoc trajectory-level filter that would already be too late.

In [ ]:
FRUIT_LIST   = ['Banana', 'Carrot', 'Guava', 'Indian_Gooseberry', 'Mango', 'Tomato']
FRUIT_TO_IDX = {f: i for i, f in enumerate(FRUIT_LIST)}
SESSION_TO_IDX = {'morning': 0, 'afternoon': 1, 'evening': 2}
LABEL_TO_IDX   = {'not_spoiled': 0, 'spoiled': 1}
IMAGE_SIZE     = 224
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
SESSION_HOUR_BUCKETS = {'morning': (5, 11), 'afternoon': (12, 16), 'evening': (16, 19)}
IR_PATTERN   = re.compile(r'(\d{8})_(\d{6})_([\d.]+)C_([\d.]+)C\.jpg$', re.IGNORECASE)
SRGB_PATTERN = re.compile(r'^(\d{8})_(\d{6})[^/]*\.jpg$', re.IGNORECASE)


def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default


def hour_to_session(hour):
    for name, (lo, hi) in SESSION_HOUR_BUCKETS.items():
        if lo <= hour < hi:
            return name
    return 'unknown'


def find_ir_folder(base):
    for name in ['IR_fusion_images', 'IR_Fusion_images', 'ir_fusion_images']:
        p = base / name
        if p.exists():
            return p
    return base / 'IR_fusion_images'


def group_images_by_session(folder, pattern):
    result = defaultdict(lambda: defaultdict(list))
    if not folder.exists():
        return {}
    for f in sorted(folder.iterdir()):
        m = pattern.search(f.name)
        if not m:
            continue
        session = hour_to_session(int(m.group(2)[:2]))
        if session != 'unknown':
            result[m.group(1)][session].append(f)
    return dict(result)


def load_image(path, transform):
    return transform(Image.open(path).convert('RGB'))


def get_rgb_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_ir_transform(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


In [ ]:
class TrimodalDataset(Dataset):
    def __init__(self, manifest_path, root, train=True, modality_dropout_prob=0.3,
                 exclude_flagged=True, split=None, gas_norm_stats_path=None,
                 cache_images=True, shared_pixel_cache=None, row_filter=None):
        self.root = Path(root); self.train = train; self.modality_dropout_prob = modality_dropout_prob
        self.rgb_transform = get_rgb_transform(train); self.ir_transform = get_ir_transform(train)
        self.gas_norm_stats = None
        if gas_norm_stats_path:
            with open(gas_norm_stats_path) as f:
                self.gas_norm_stats = json.load(f)['stats']
        with open(manifest_path, newline='') as f:
            all_rows = list(csv.DictReader(f))
        if exclude_flagged:
            all_rows = [r for r in all_rows if r.get('exclude', '').strip().lower() != 'true']
        if split:
            all_rows = [r for r in all_rows if r.get('split', '').strip().lower() == split.lower()]
        if row_filter is not None:
            all_rows = [r for r in all_rows if row_filter(r)]
        self.samples = []; self._build_samples(all_rows)
        self._image_cache = {}; self._build_image_index()
        self._pixel_cache = {}
        if shared_pixel_cache is not None:
            self._pixel_cache = shared_pixel_cache
        elif cache_images:
            self._warmup_pixel_cache()
        print(f'TrimodalDataset: {len(self.samples)} sessions '
              f'(split={split}, row_filter={row_filter is not None})')

    def _build_samples(self, rows):
        for row in rows:
            fruit = row['fruit']; label = row['label']; date_str = row['actual_date']
            global_day = int(row['corrected_day_index'])
            days_until = safe_float(row.get('days_until_spoilage', ''), default=-1.0)
            for session in ['morning', 'afternoon', 'evening']:
                si = SESSION_TO_IDX[session]
                ir_avail = safe_float(row.get(f'ir_{session}_count', 0)) > 0
                ir_tmin = safe_float(row.get(f'ir_{session}_tmin', ''), 0.0)
                ir_tmax = safe_float(row.get(f'ir_{session}_tmax', ''), 0.0)
                ir_trange = safe_float(row.get(f'ir_{session}_trange', ''), 0.0)
                srgb_avail = safe_float(row.get(f'srgb_{session}_count', 0)) > 0
                gas_avail = row.get(f'methane_{session}_present', '').strip().upper() == 'TRUE'
                mean_ppm = safe_float(row.get(f'methane_{session}_ppm', ''), 0.0)
                std_ppm = safe_float(row.get(f'methane_{session}_std', ''), 0.0)
                left_ppm = safe_float(row.get(f'methane_{session}_left', ''), 0.0)
                right_ppm = safe_float(row.get(f'methane_{session}_right', ''), 0.0)
                if not ir_avail and not srgb_avail and not gas_avail:
                    continue
                self.samples.append({'fruit': fruit, 'label': label, 'actual_date': date_str,
                    'corrected_day_index': global_day, 'session': session, 'session_idx': si,
                    'fruit_idx': FRUIT_TO_IDX.get(fruit, 0), 'label_idx': LABEL_TO_IDX.get(label, 0),
                    'days_until_spoilage': days_until, 'ir_tmin': ir_tmin, 'ir_tmax': ir_tmax,
                    'ir_trange': ir_trange, 'gas_mean': mean_ppm, 'gas_std': std_ppm,
                    'gas_left': left_ppm, 'gas_right': right_ppm, 'gas_asym': abs(left_ppm - right_ppm),
                    'rgb_available': srgb_avail, 'ir_available': ir_avail, 'gas_available': gas_avail,
                    'split': row.get('split', '').strip().lower()})

    def _build_image_index(self):
        label_map = {'not_spoiled': 'Not_spoiled', 'spoiled': 'Spoiled'}
        for fruit in FRUIT_LIST:
            for lk, lf in label_map.items():
                base = self.root / 'Classified' / fruit / lf
                for d, ss in group_images_by_session(base / 'sRGB_images', SRGB_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['rgb'] = ps
                for d, ss in group_images_by_session(find_ir_folder(base), IR_PATTERN).items():
                    for s, ps in ss.items():
                        self._image_cache.setdefault((fruit, lk, d, s), {'rgb': [], 'ir': []})['ir'] = ps
        banana_ir = group_images_by_session(find_ir_folder(self.root / 'Normal' / 'Banana'), IR_PATTERN)
        spoiled_dates = {date for (fr, lb, date, _) in self._image_cache if fr == 'Banana' and lb == 'spoiled'}
        for d, ss in banana_ir.items():
            if d in spoiled_dates:
                for s, ps in ss.items():
                    key = ('Banana', 'spoiled', d, s)
                    self._image_cache.setdefault(key, {'rgb': [], 'ir': []})
                    if not self._image_cache[key].get('ir'):
                        self._image_cache[key]['ir'] = ps

    def _load_cached(self, path, transform):
        cached = self._pixel_cache.get(str(path))
        if cached is not None:
            return transform(Image.fromarray(cached.permute(1, 2, 0).numpy()))
        return load_image(path, transform)

    def _warmup_pixel_cache(self):
        all_paths = set()
        for v in self._image_cache.values():
            all_paths.update(v.get('rgb', [])); all_paths.update(v.get('ir', []))
        for path in all_paths:
            try:
                img = self.rgb_transform.transforms[0](Image.open(path).convert('RGB'))
                self._pixel_cache[str(path)] = torch.from_numpy(np.array(img)).permute(2, 0, 1)
            except Exception:
                pass

    def _normalize_gas(self, fruit, session, mean_ppm, std_ppm, left_ppm, right_ppm):
        if self.gas_norm_stats is None:
            return mean_ppm, std_ppm, left_ppm, right_ppm
        s = self.gas_norm_stats.get(fruit, {}).get(session, {})

        def norm(val, key):
            st = s.get(key, {'mean': 0., 'std': 1.})
            return (val - st['mean']) / st['std']
        return norm(mean_ppm, 'mean_ppm'), norm(std_ppm, 'std_ppm'), norm(left_ppm, 'left_ppm'), norm(right_ppm, 'right_ppm')

    def _apply_dropout(self, rgb_avail, ir_avail, gas_avail):
        if not self.train:
            return rgb_avail, ir_avail, gas_avail
        while True:
            r = rgb_avail and (random.random() > self.modality_dropout_prob)
            i = ir_avail and (random.random() > self.modality_dropout_prob)
            g = gas_avail and (random.random() > self.modality_dropout_prob)
            if r or i or g:
                return r, i, g

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fruit = s['fruit']; label = s['label']; date_str = s['actual_date']
        session = s['session']; si = s['session_idx']
        rgb_avail, ir_avail, gas_avail = self._apply_dropout(s['rgb_available'], s['ir_available'], s['gas_available'])
        cached = self._image_cache.get((fruit, label, date_str, session), {'rgb': [], 'ir': []})
        if rgb_avail and cached['rgb']:
            rgb_tensors = torch.stack([self._load_cached(p, self.rgb_transform) for p in cached['rgb']])
        else:
            rgb_avail = False; rgb_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        if ir_avail and cached['ir']:
            ir_tensors = torch.stack([self._load_cached(p, self.ir_transform) for p in cached['ir']])
        else:
            ir_avail = False; ir_tensors = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
        if gas_avail:
            m, sd, l, r = self._normalize_gas(fruit, session, s['gas_mean'], s['gas_std'], s['gas_left'], s['gas_right'])
            gas = torch.tensor([m, sd, l, r, abs(l - r), float(si) / 2.], dtype=torch.float32)
        else:
            gas = torch.zeros(6)
        return {'rgb_images': rgb_tensors, 'ir_images': ir_tensors, 'gas': gas,
                'rgb_available': torch.tensor(rgb_avail, dtype=torch.bool),
                'ir_available': torch.tensor(ir_avail, dtype=torch.bool),
                'gas_available': torch.tensor(gas_avail, dtype=torch.bool),
                'fruit_idx': torch.tensor(s['fruit_idx'], dtype=torch.long),
                'session_idx': torch.tensor(si, dtype=torch.long),
                'label': torch.tensor(s['label_idx'], dtype=torch.long),
                'days_until_spoilage': torch.tensor(s['days_until_spoilage'], dtype=torch.float32),
                'fruit': fruit, 'actual_date': date_str, 'corrected_day_index': s['corrected_day_index'],
                'split': s['split']}


In [ ]:
class DayLevelSequenceDataset(Dataset):
    def __init__(self, manifest_path=MANIFEST, root=ROOT, split="train",
                 gas_norm_stats_path=GAS_NORM, shared_pixel_cache=None, row_filter=None):
        base = TrimodalDataset(
            manifest_path, root, train=False, modality_dropout_prob=0.0,
            split=split, gas_norm_stats_path=gas_norm_stats_path, cache_images=True,
            shared_pixel_cache=shared_pixel_cache, row_filter=row_filter,
        )
        self._pixel_cache = base._pixel_cache
        day_groups = defaultdict(list)
        for i in range(len(base)):
            item = base[i]
            key = (item["fruit"], int(item["label"].item()), int(item["corrected_day_index"]))
            day_groups[key].append(item)
        day_entries = {}
        for (fruit, label, day_idx), sessions in day_groups.items():
            sessions.sort(key=lambda s: int(s["session_idx"].item()))
            rgb_img, ir_img = None, None
            for s in sessions:
                if rgb_img is None and bool(s["rgb_available"].item()) and len(s["rgb_images"]) > 0:
                    rgb_img = s["rgb_images"][0]
                if ir_img is None and bool(s["ir_available"].item()) and len(s["ir_images"]) > 0:
                    ir_img = s["ir_images"][0]
            gas_vals = [s["gas"] for s in sessions if bool(s["gas_available"].item())]
            gas_avail = len(gas_vals) > 0
            gas_vec = torch.stack(gas_vals).mean(dim=0) if gas_avail else torch.zeros(6)
            day_entries.setdefault((fruit, label), []).append({
                "day_idx": day_idx, "rgb_img": rgb_img, "ir_img": ir_img,
                "gas_vec": gas_vec, "gas_avail": gas_avail,
                "fruit_idx": sessions[0]["fruit_idx"], "label": sessions[0]["label"],
                "days_until": sessions[0]["days_until_spoilage"],
                "split": sessions[0]["split"],
            })
        self.trajectories = []
        for (fruit, label), days in day_entries.items():
            days.sort(key=lambda d: d["day_idx"])
            last_observed_idx = None
            for d in days:
                d["delta"] = 0.0 if last_observed_idx is None else float(d["day_idx"] - last_observed_idx)
                if d["gas_avail"]:
                    last_observed_idx = d["day_idx"]
            self.trajectories.append({"fruit": fruit, "label": label, "days": days})
        n_days_total = sum(len(t["days"]) for t in self.trajectories)
        print(f"  DayLevelSequenceDataset: {len(self.trajectories)} trajectories, "
              f"{n_days_total} day-entries (split={split}, row_filter={row_filter is not None})")

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        traj = self.trajectories[idx]["days"]
        T = len(traj)
        blank_rgb = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        blank_ir = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
        rgb_seq = torch.stack([d["rgb_img"] if d["rgb_img"] is not None else blank_rgb for d in traj])
        ir_seq = torch.stack([d["ir_img"] if d["ir_img"] is not None else blank_ir for d in traj])
        rgb_avail = torch.tensor([d["rgb_img"] is not None for d in traj], dtype=torch.bool)
        ir_avail = torch.tensor([d["ir_img"] is not None for d in traj], dtype=torch.bool)
        gas_seq = torch.stack([d["gas_vec"] for d in traj])
        gas_mask = torch.tensor([d["gas_avail"] for d in traj], dtype=torch.float32)
        gas_delta = torch.tensor([d["delta"] for d in traj], dtype=torch.float32)
        return {"rgb_seq": rgb_seq, "ir_seq": ir_seq, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
                "gas_seq": gas_seq, "gas_mask": gas_mask, "gas_delta": gas_delta, "T": T,
                "fruit_idx": traj[0]["fruit_idx"], "label": traj[-1]["label"],
                "days_until": traj[-1]["days_until"],
                "day_splits": [d["split"] for d in traj]}


def day_sequence_collate(batch):
    max_T = max(b["T"] for b in batch)
    B = len(batch)
    rgb = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE)
    ir = torch.zeros(B, max_T, 3, IMAGE_SIZE, IMAGE_SIZE)
    rgb_avail = torch.zeros(B, max_T, dtype=torch.bool)
    ir_avail = torch.zeros(B, max_T, dtype=torch.bool)
    gas = torch.zeros(B, max_T, 6)
    gas_mask = torch.zeros(B, max_T)
    gas_delta = torch.zeros(B, max_T)
    pad_mask = torch.ones(B, max_T, dtype=torch.bool)
    fruit_idx = torch.zeros(B, dtype=torch.long)
    label = torch.zeros(B, dtype=torch.long)
    days_until = torch.zeros(B, dtype=torch.float32)
    for i, b in enumerate(batch):
        T = b["T"]
        rgb[i, :T] = b["rgb_seq"]; ir[i, :T] = b["ir_seq"]
        rgb_avail[i, :T] = b["rgb_avail"]; ir_avail[i, :T] = b["ir_avail"]
        gas[i, :T] = b["gas_seq"]; gas_mask[i, :T] = b["gas_mask"]; gas_delta[i, :T] = b["gas_delta"]
        pad_mask[i, :T] = False
        fruit_idx[i] = b["fruit_idx"]; label[i] = b["label"]; days_until[i] = b["days_until"]
    return {"rgb_seq": rgb, "ir_seq": ir, "rgb_avail": rgb_avail, "ir_avail": ir_avail,
            "gas_seq": gas, "gas_mask": gas_mask, "gas_delta": gas_delta, "pad_mask": pad_mask,
            "fruit_idx": fruit_idx, "label": label, "days_until": days_until}


class OfflineAugmentedDayLevelSequenceDataset(Dataset):
    def __init__(self, base_dataset, transform_type="flip_rotation", n_copies=1):
        assert transform_type == "flip_rotation"
        self.base = base_dataset
        self.n_copies = n_copies
        self.index_map = []
        for traj_idx in range(len(base_dataset)):
            self.index_map.append((traj_idx, 0))
            for copy_id in range(1, n_copies + 1):
                self.index_map.append((traj_idx, copy_id))
        self.trajectories = [base_dataset.trajectories[traj_idx] for traj_idx, _ in self.index_map]

    def __len__(self):
        return len(self.index_map)

    def _augment_image(self, img, seed):
        g = torch.Generator().manual_seed(seed)
        out = torch.flip(img, dims=[-1])
        angle = (torch.rand(1, generator=g).item() * 30.0) - 15.0
        return TF.rotate(out, angle)

    def __getitem__(self, idx):
        traj_idx, copy_id = self.index_map[idx]
        item = self.base[traj_idx]
        if copy_id == 0:
            return item
        seed_base = traj_idx * 10_000 + copy_id * 100
        item = dict(item)
        item["rgb_seq"] = torch.stack([self._augment_image(item["rgb_seq"][t], seed_base + t)
                                        for t in range(item["rgb_seq"].shape[0])])
        item["ir_seq"] = torch.stack([self._augment_image(item["ir_seq"][t], seed_base + t)
                                       for t in range(item["ir_seq"].shape[0])])
        return item


## Model building blocks
*(Reused verbatim from `04_fusion_model.ipynb` / `05_ablation_study.ipynb`. Both `TrimodalFusionModel` (Part B/C, matches `04`'s exact save format) and the `AblationModel` family (Part A, matches `05`'s 7 configs) are needed here, since this notebook works across both notebooks' checkpoints.)*

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False), nn.ReLU(inplace=True),
                                  nn.Conv2d(hidden, channels, 1, bias=False))

    def forward(self, x):
        return torch.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out, _ = x.max(dim=1, keepdim=True)
        return torch.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x = x * self.channel_attn(x)
        sa = self.spatial_attn(x)
        return x * sa, sa


BACKBONE_FEATURE_DIMS = {"efficientnet_b0": 1280}


def _infer_feature_dim(backbone, img_size=IMAGE_SIZE):
    backbone.eval()
    with torch.no_grad():
        out = backbone(torch.zeros(1, 3, img_size, img_size))
    return out.shape[1]


def build_cnn_backbone(name: str):
    if name == "efficientnet_b0":
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        return base.features, BACKBONE_FEATURE_DIMS[name]
    elif name == "convnext_tiny":
        base = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        backbone = base.features
        feat_dim = _infer_feature_dim(backbone)
        BACKBONE_FEATURE_DIMS[name] = feat_dim
        return backbone, feat_dim
    else:
        raise ValueError(f"Unknown backbone: {name}")


class VisualEncoderToggle(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", freeze_backbone=True, use_cbam=True, chunk_size=8):
        super().__init__()
        self.use_cbam = use_cbam
        self.backbone, self.feature_dim = build_cnn_backbone(backbone_name)
        self.chunk_size = chunk_size
        if use_cbam:
            self.cbam = CBAM(self.feature_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def unfreeze_last_n_layers(self, n: int):
        self.freeze_backbone()
        for child in list(self.backbone.children())[-n:]:
            for p in child.parameters():
                p.requires_grad = True

    def forward(self, seq: torch.Tensor):
        B, T, C, H, W = seq.shape
        flat = seq.reshape(B * T, C, H, W)
        feats_list = []
        needs_ckpt = (torch.is_grad_enabled() and any(p.requires_grad for p in self.backbone.parameters()))
        for start in range(0, flat.shape[0], self.chunk_size):
            chunk = flat[start:start + self.chunk_size]
            feat_map = checkpoint(self.backbone, chunk, use_reentrant=False) if needs_ckpt else self.backbone(chunk)
            if self.use_cbam:
                feat_map, _ = self.cbam(feat_map)
            feats_list.append(self.pool(feat_map).flatten(1))
        feats = torch.cat(feats_list, dim=0).view(B, T, -1)
        return feats


class GRUDCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, x_mean=None):
        super().__init__()
        self.input_dim = input_dim; self.hidden_dim = hidden_dim
        if x_mean is None:
            x_mean = [0.0] * input_dim
        self.register_buffer("x_mean", torch.as_tensor(x_mean, dtype=torch.float32))
        self.W_gamma_x = nn.Linear(1, input_dim)
        self.W_gamma_h = nn.Linear(1, hidden_dim)
        self.gru_cell = nn.GRUCell(input_dim * 2, hidden_dim)

    def forward(self, x_t, m_t, delta_t, x_last, h_prev):
        gamma_x = torch.exp(-torch.clamp(self.W_gamma_x(delta_t), min=0.0))
        x_bar = self.x_mean.unsqueeze(0).expand_as(x_t)
        x_hat = m_t * x_t + (1 - m_t) * (gamma_x * x_last + (1 - gamma_x) * x_bar)
        gamma_h = torch.exp(-torch.clamp(self.W_gamma_h(delta_t), min=0.0))
        h_t = self.gru_cell(torch.cat([x_hat, m_t], dim=-1), gamma_h * h_prev)
        return h_t, x_hat


class GasGRUDSequenceEncoder(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64, x_mean=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell = GRUDCell(input_dim, hidden_dim, x_mean)

    def forward(self, gas_seq, gas_mask, gas_delta, pad_mask):
        B, T, D = gas_seq.shape
        device = gas_seq.device
        h = torch.zeros(B, self.hidden_dim, device=device)
        x_last = torch.zeros(B, D, device=device)
        h_seq = []
        for t in range(T):
            x_t = gas_seq[:, t]
            m_t = gas_mask[:, t].unsqueeze(-1).expand(-1, D)
            delta_t = gas_delta[:, t].unsqueeze(-1)
            valid_t = (~pad_mask[:, t]).float().unsqueeze(-1)
            h_new, x_hat = self.cell(x_t, m_t, delta_t, x_last, h)
            h = valid_t * h_new + (1 - valid_t) * h
            x_last = torch.where(m_t.bool(), x_t, x_last)
            h_seq.append(h)
        return torch.stack(h_seq, dim=1)


class GasEncoderMeanImpute(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gru = nn.GRU(input_dim + 1, hidden_dim, batch_first=True)

    def forward(self, gas_seq, gas_mask, gas_delta, pad_mask):
        imputed = gas_seq * gas_mask.unsqueeze(-1)
        gru_input = torch.cat([imputed, gas_mask.unsqueeze(-1)], dim=-1)
        lengths = (~pad_mask).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(gru_input, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=gas_seq.shape[1])
        return out


class CrossModalFusion(nn.Module):
    def __init__(self, d_model=64, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, rgb_t, ir_t, gas_t, rgb_avail_t, ir_avail_t, gas_avail_t):
        tokens = torch.stack([rgb_t, ir_t, gas_t], dim=1)
        avail = torch.stack([rgb_avail_t, ir_avail_t, gas_avail_t], dim=1)
        key_padding_mask = ~avail
        fully_missing = key_padding_mask.all(dim=1)
        if fully_missing.any():
            key_padding_mask = key_padding_mask.clone()
            key_padding_mask[fully_missing] = False
        attended, attn_w = self.attn(tokens, tokens, tokens, key_padding_mask=key_padding_mask,
                                      need_weights=True, average_attn_weights=True)
        out = self.norm(tokens + attended)
        return out.mean(dim=1), attn_w


class FusionToggle(nn.Module):
    def __init__(self, d_model=64, num_heads=4, dropout=0.1, use_cross_attn=True):
        super().__init__()
        self.use_cross_attn = use_cross_attn
        if use_cross_attn:
            self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_model)
        else:
            self.concat_proj = nn.Sequential(nn.Linear(d_model * 3, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, rgb_t, ir_t, gas_t, rgb_avail_t, ir_avail_t, gas_avail_t):
        if self.use_cross_attn:
            tokens = torch.stack([rgb_t, ir_t, gas_t], dim=1)
            avail = torch.stack([rgb_avail_t, ir_avail_t, gas_avail_t], dim=1)
            key_padding_mask = ~avail
            fully_missing = key_padding_mask.all(dim=1)
            if fully_missing.any():
                key_padding_mask = key_padding_mask.clone()
                key_padding_mask[fully_missing] = False
            attended, attn_w = self.attn(tokens, tokens, tokens, key_padding_mask=key_padding_mask,
                                          need_weights=True, average_attn_weights=True)
            out = self.norm(tokens + attended)
            return out.mean(dim=1), attn_w
        else:
            concat = torch.cat([rgb_t, ir_t, gas_t], dim=-1)
            return self.concat_proj(concat), None


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2, dim_feedforward=512, dropout=0.1, max_len=100):
        super().__init__()
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, z_seq, pad_mask):
        z_seq = self.pos_enc(z_seq)
        return self.encoder(z_seq, src_key_padding_mask=pad_mask)


class LSTMTemporalModule(nn.Module):
    def __init__(self, d_model=64, hidden_dim=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(d_model, hidden_dim, num_layers=num_layers, batch_first=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.out_proj = nn.Linear(hidden_dim, d_model) if hidden_dim != d_model else nn.Identity()

    def forward(self, z_seq, pad_mask):
        lengths = (~pad_mask).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(z_seq, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=z_seq.shape[1])
        return self.out_proj(out)


def gather_last_valid(H, pad_mask):
    device = H.device
    lengths = (~pad_mask).sum(dim=1)
    last_idx = (lengths - 1).clamp(min=0)
    B = H.shape[0]
    return H[torch.arange(B, device=device), last_idx], last_idx


class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, n_tasks: int = 3):
        super().__init__()
        self.log_sigma = nn.Parameter(torch.zeros(n_tasks))

    def forward(self, losses: list):
        total = 0.0
        for i, L_i in enumerate(losses):
            precision = torch.exp(-2 * self.log_sigma[i])
            total = total + 0.5 * precision * L_i + self.log_sigma[i]
        return total


## `TrimodalFusionModel` (Part B/C, matches `04`'s save format)

In [ ]:
class TrimodalFusionModel(nn.Module):
    def __init__(self, rgb_backbone_name="convnext_tiny", ir_backbone_name="efficientnet_b0",
                 d_model=64, gas_hidden=64, num_heads=4, temporal_layers=2,
                 dropout=0.5, cls_dropout=0.5, freeze_visual_backbone=True):
        super().__init__()
        self.d_model = d_model
        self.rgb_encoder = VisualEncoderToggle(rgb_backbone_name, freeze_visual_backbone, use_cbam=True)
        self.ir_encoder = VisualEncoderToggle(ir_backbone_name, freeze_visual_backbone, use_cbam=True)
        self.gas_encoder = GasGRUDSequenceEncoder(input_dim=6, hidden_dim=gas_hidden)

        self.rgb_proj = nn.Sequential(nn.Linear(self.rgb_encoder.feature_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.ir_proj = nn.Sequential(nn.Linear(self.ir_encoder.feature_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.gas_proj = nn.Sequential(nn.Linear(gas_hidden, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))

        self.fusion = CrossModalFusion(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.temporal = TemporalTransformer(d_model=d_model, nhead=num_heads, num_layers=temporal_layers, dropout=dropout)

        self.cls_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 2))
        self.reg_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 1), nn.ReLU())
        self.gas_recon_head = nn.Sequential(nn.Linear(d_model * 2, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 6))

    def unfreeze_visual_last_n(self, n: int):
        self.rgb_encoder.unfreeze_last_n_layers(n); self.ir_encoder.unfreeze_last_n_layers(n)

    def unfreeze_visual_full(self):
        self.rgb_encoder.unfreeze_backbone(); self.ir_encoder.unfreeze_backbone()

    def forward(self, batch: dict):
        device = next(self.parameters()).device
        rgb_seq = batch["rgb_seq"].to(device); ir_seq = batch["ir_seq"].to(device)
        rgb_avail = batch["rgb_avail"].to(device); ir_avail = batch["ir_avail"].to(device)
        gas_seq = batch["gas_seq"].to(device); gas_mask = batch["gas_mask"].to(device)
        gas_delta = batch["gas_delta"].to(device); pad_mask = batch["pad_mask"].to(device)
        B, T = rgb_avail.shape

        rgb_feat = self.rgb_encoder(rgb_seq); ir_feat = self.ir_encoder(ir_seq)
        gas_feat = self.gas_encoder(gas_seq, gas_mask, gas_delta, pad_mask)

        rgb_proj = self.rgb_proj(rgb_feat) * rgb_avail.unsqueeze(-1).float()
        ir_proj = self.ir_proj(ir_feat) * ir_avail.unsqueeze(-1).float()
        gas_proj = self.gas_proj(gas_feat) * gas_mask.unsqueeze(-1)

        rgb_flat = rgb_proj.reshape(B * T, -1); ir_flat = ir_proj.reshape(B * T, -1); gas_flat = gas_proj.reshape(B * T, -1)
        rgb_av_flat = rgb_avail.reshape(B * T); ir_av_flat = ir_avail.reshape(B * T); gas_av_flat = gas_mask.reshape(B * T).bool()
        z_flat, _ = self.fusion(rgb_flat, ir_flat, gas_flat, rgb_av_flat, ir_av_flat, gas_av_flat)
        z_seq = z_flat.reshape(B, T, -1)

        H = self.temporal(z_seq, pad_mask)
        H_T, last_idx = gather_last_valid(H, pad_mask)
        cls_logits = self.cls_head(H_T); reg_output = self.reg_head(H_T)

        rgb_last = rgb_proj[torch.arange(B, device=device), last_idx]
        ir_last = ir_proj[torch.arange(B, device=device), last_idx]
        gas_recon = self.gas_recon_head(torch.cat([rgb_last, ir_last], dim=1))

        return {"cls_logits": cls_logits, "reg_output": reg_output, "gas_recon": gas_recon, "last_idx": last_idx}


## `AblationModel` / `AblationModelCustomRGB` family (Part A, matches `05`'s 7 configs)

In [ ]:
ABLATION_CONFIGS = {
    "baseline_a_early": {"fusion_type": "early", "use_cbam": False, "use_cross_attn": False,
                          "temporal_type": "lstm", "missing_handling": "mean"},
    "baseline_b_late": {"fusion_type": "late", "use_cbam": False, "use_cross_attn": False,
                         "temporal_type": "lstm", "missing_handling": "mean"},
    "ablation_1_no_cbam": {"fusion_type": "intermediate", "use_cbam": False, "use_cross_attn": True,
                            "temporal_type": "transformer", "missing_handling": "grud"},
    "ablation_2_no_cross_attn": {"fusion_type": "intermediate", "use_cbam": True, "use_cross_attn": False,
                                  "temporal_type": "transformer", "missing_handling": "grud"},
    "ablation_3_lstm_not_transformer": {"fusion_type": "intermediate", "use_cbam": True, "use_cross_attn": True,
                                         "temporal_type": "lstm", "missing_handling": "grud"},
    "ablation_4_mean_impute_not_grud": {"fusion_type": "intermediate", "use_cbam": True, "use_cross_attn": True,
                                         "temporal_type": "transformer", "missing_handling": "mean"},
    "full_model": {"fusion_type": "intermediate", "use_cbam": True, "use_cross_attn": True,
                   "temporal_type": "transformer", "missing_handling": "grud"},
}
CONFIG_ORDER = ["full_model", "ablation_1_no_cbam", "ablation_2_no_cross_attn",
                "ablation_3_lstm_not_transformer", "ablation_4_mean_impute_not_grud",
                "baseline_a_early", "baseline_b_late"]
CONFIG_LABELS = {
    "full_model": "Full Model", "ablation_1_no_cbam": "Ablation 1 (no CBAM)",
    "ablation_2_no_cross_attn": "Ablation 2 (no cross-attn)",
    "ablation_3_lstm_not_transformer": "Ablation 3 (LSTM)",
    "ablation_4_mean_impute_not_grud": "Ablation 4 (mean-impute)",
    "baseline_a_early": "Baseline A (early fusion)", "baseline_b_late": "Baseline B (late fusion)",
}


class AblationModel(nn.Module):
    def __init__(self, config: dict, d_model=64, backbone_name="efficientnet_b0", gas_hidden=64,
                 num_heads=4, temporal_layers=2, dropout=0.5, cls_dropout=0.5, freeze_visual_backbone=True):
        super().__init__()
        self.config = config; self.d_model = d_model
        self.fusion_type = config["fusion_type"]

        self.rgb_encoder = VisualEncoderToggle(backbone_name, freeze_visual_backbone, use_cbam=config["use_cbam"])
        self.ir_encoder = VisualEncoderToggle(backbone_name, freeze_visual_backbone, use_cbam=config["use_cbam"])
        self.gas_encoder = (GasGRUDSequenceEncoder(input_dim=6, hidden_dim=gas_hidden) if config["missing_handling"] == "grud"
                             else GasEncoderMeanImpute(input_dim=6, hidden_dim=gas_hidden))

        self.rgb_proj = nn.Sequential(nn.Linear(self.rgb_encoder.feature_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.ir_proj = nn.Sequential(nn.Linear(self.ir_encoder.feature_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
        self.gas_proj = nn.Sequential(nn.Linear(gas_hidden, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))

        def make_temporal():
            if config["temporal_type"] == "transformer":
                return TemporalTransformer(d_model=d_model, nhead=num_heads, num_layers=temporal_layers, dropout=dropout)
            return LSTMTemporalModule(d_model=d_model, hidden_dim=d_model, dropout=dropout)

        def make_heads():
            cls = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 2))
            reg = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 1), nn.ReLU())
            return cls, reg

        if self.fusion_type == "intermediate":
            self.fusion = FusionToggle(d_model=d_model, num_heads=num_heads, dropout=dropout, use_cross_attn=config["use_cross_attn"])
            self.temporal = make_temporal(); self.cls_head, self.reg_head = make_heads()
        elif self.fusion_type == "early":
            self.early_proj = nn.Sequential(nn.Linear(d_model * 3, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))
            self.temporal = make_temporal(); self.cls_head, self.reg_head = make_heads()
        elif self.fusion_type == "late":
            self.temporal_rgb = make_temporal(); self.temporal_ir = make_temporal(); self.temporal_gas = make_temporal()
            self.cls_head_rgb, self.reg_head_rgb = make_heads()
            self.cls_head_ir, self.reg_head_ir = make_heads()
            self.cls_head_gas, self.reg_head_gas = make_heads()
        else:
            raise ValueError(f"Unknown fusion_type: {self.fusion_type}")
        self.gas_recon_head = nn.Sequential(nn.Linear(d_model * 2, 128), nn.ReLU(), nn.Dropout(cls_dropout), nn.Linear(128, 6))

    def forward(self, batch: dict):
        device = next(self.parameters()).device
        rgb_seq = batch["rgb_seq"].to(device); ir_seq = batch["ir_seq"].to(device)
        rgb_avail = batch["rgb_avail"].to(device); ir_avail = batch["ir_avail"].to(device)
        gas_seq = batch["gas_seq"].to(device); gas_mask = batch["gas_mask"].to(device)
        gas_delta = batch["gas_delta"].to(device); pad_mask = batch["pad_mask"].to(device)
        B, T = rgb_avail.shape

        rgb_feat = self.rgb_encoder(rgb_seq); ir_feat = self.ir_encoder(ir_seq)
        gas_feat = self.gas_encoder(gas_seq, gas_mask, gas_delta, pad_mask)

        rgb_proj = self.rgb_proj(rgb_feat) * rgb_avail.unsqueeze(-1).float()
        ir_proj = self.ir_proj(ir_feat) * ir_avail.unsqueeze(-1).float()
        gas_proj = self.gas_proj(gas_feat) * gas_mask.unsqueeze(-1)

        if self.fusion_type == "intermediate":
            rgb_flat = rgb_proj.reshape(B * T, -1); ir_flat = ir_proj.reshape(B * T, -1); gas_flat = gas_proj.reshape(B * T, -1)
            rgb_av_flat = rgb_avail.reshape(B * T); ir_av_flat = ir_avail.reshape(B * T); gas_av_flat = gas_mask.reshape(B * T).bool()
            z_flat, _ = self.fusion(rgb_flat, ir_flat, gas_flat, rgb_av_flat, ir_av_flat, gas_av_flat)
            z_seq = z_flat.reshape(B, T, -1)
            H = self.temporal(z_seq, pad_mask)
            H_T, last_idx = gather_last_valid(H, pad_mask)
            cls_logits = self.cls_head(H_T); reg_output = self.reg_head(H_T)
        elif self.fusion_type == "early":
            z_seq = self.early_proj(torch.cat([rgb_proj, ir_proj, gas_proj], dim=-1))
            H = self.temporal(z_seq, pad_mask)
            H_T, last_idx = gather_last_valid(H, pad_mask)
            cls_logits = self.cls_head(H_T); reg_output = self.reg_head(H_T)
        else:
            H_rgb = self.temporal_rgb(rgb_proj, pad_mask); H_ir = self.temporal_ir(ir_proj, pad_mask); H_gas = self.temporal_gas(gas_proj, pad_mask)
            HT_rgb, last_idx = gather_last_valid(H_rgb, pad_mask)
            HT_ir, _ = gather_last_valid(H_ir, pad_mask); HT_gas, _ = gather_last_valid(H_gas, pad_mask)
            logits_rgb = self.cls_head_rgb(HT_rgb); logits_ir = self.cls_head_ir(HT_ir); logits_gas = self.cls_head_gas(HT_gas)
            reg_rgb = self.reg_head_rgb(HT_rgb); reg_ir = self.reg_head_ir(HT_ir); reg_gas = self.reg_head_gas(HT_gas)
            rgb_av_last = rgb_avail[torch.arange(B, device=device), last_idx].float()
            ir_av_last = ir_avail[torch.arange(B, device=device), last_idx].float()
            gas_av_last = gas_mask[torch.arange(B, device=device), last_idx]
            avail_stack = torch.stack([rgb_av_last, ir_av_last, gas_av_last], dim=1)
            denom = avail_stack.sum(dim=1, keepdim=True).clamp(min=1.0)
            logits_stack = torch.stack([logits_rgb, logits_ir, logits_gas], dim=1)
            cls_logits = (logits_stack * avail_stack.unsqueeze(-1)).sum(dim=1) / denom
            reg_stack = torch.stack([reg_rgb, reg_ir, reg_gas], dim=1).squeeze(-1)
            reg_output = ((reg_stack * avail_stack).sum(dim=1) / denom.squeeze(-1)).unsqueeze(-1)

        rgb_last = rgb_proj[torch.arange(B, device=device), last_idx]
        ir_last = ir_proj[torch.arange(B, device=device), last_idx]
        gas_recon = self.gas_recon_head(torch.cat([rgb_last, ir_last], dim=1))
        return {"cls_logits": cls_logits, "reg_output": reg_output, "gas_recon": gas_recon, "last_idx": last_idx}


class AblationModelCustomRGB(AblationModel):
    def __init__(self, config, **kwargs):
        rgb_encoder_factory = kwargs.pop("rgb_encoder_factory")
        super().__init__(config, **kwargs)
        d_model = self.d_model
        dropout = kwargs.get("dropout", 0.5)
        self.rgb_encoder = rgb_encoder_factory()
        self.rgb_proj = nn.Sequential(nn.Linear(self.rgb_encoder.feature_dim, d_model), nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))


print("TrimodalFusionModel, AblationModel, AblationModelCustomRGB ready.")


## Training / evaluation helpers
*(Reused verbatim from `04_fusion_model.ipynb` / `05_ablation_study.ipynb`. `run_epoch` here supports `train=True` for Part B's fresh k-fold training, unlike the read-only collection helpers used in Part A and Part C.)*

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def build_weighted_sampler(dataset):
    labels = [int(t["label"]) for t in dataset.trajectories]
    n_total = len(labels); n_spoiled = max(sum(labels), 1); n_fresh = max(n_total - sum(labels), 1)
    w_spoiled = n_total / (2 * n_spoiled); w_fresh = n_total / (2 * n_fresh)
    weights = [w_spoiled if l == 1 else w_fresh for l in labels]
    return WeightedRandomSampler(weights=weights, num_samples=n_total, replacement=True)


def calibrate_threshold(scores, labels, thresholds=None):
    if thresholds is None:
        thresholds = THRESHOLDS_TO_SWEEP
    if len(set(labels)) < 2:
        return 0.5, 0.0
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = [1 if s >= t else 0 for s in scores]
        f = f1_score(labels, preds, average="binary", zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def collect_probs_labels(model, loader):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch)
            probs = torch.softmax(out["cls_logits"], dim=1).cpu().numpy()
            all_labels.extend(batch["label"].cpu().numpy().tolist())
            all_probs.extend(probs.tolist())
    return np.array(all_probs), all_labels


def get_calibrated_threshold(model, calib_loader):
    probs, labels = collect_probs_labels(model, calib_loader)
    thr, _ = calibrate_threshold(probs[:, 1].tolist(), labels)
    return thr


def compute_loss(out, batch, uncertainty_loss, cfg, device):
    labels = batch["label"].to(device)
    days = batch["days_until"].to(device)
    class_weights = torch.tensor(cfg["cls_class_weights"], device=device)
    cls_loss = nn.CrossEntropyLoss(weight=class_weights)(out["cls_logits"], labels)
    reg_loss = nn.SmoothL1Loss()(out["reg_output"].squeeze(1), days)
    pad_mask = batch["pad_mask"].to(device); gas_mask = batch["gas_mask"].to(device); gas_seq = batch["gas_seq"].to(device)
    last_idx = out["last_idx"]; B = labels.shape[0]
    gas_avail_last = gas_mask[torch.arange(B, device=device), last_idx].bool()
    gas_true_last = gas_seq[torch.arange(B, device=device), last_idx]
    recon_loss = (nn.MSELoss()(out["gas_recon"][gas_avail_last], gas_true_last[gas_avail_last])
                  if gas_avail_last.any() else torch.zeros((), device=device))
    total_loss = uncertainty_loss([cls_loss, reg_loss, recon_loss])
    return total_loss, {"cls_loss": cls_loss.item(), "reg_loss": reg_loss.item(), "recon_loss": recon_loss.item()}


def run_epoch(model, uncertainty_loss, loader, optimizer, cfg, train=True, threshold=None):
    model.train() if train else model.eval()
    uncertainty_loss.train() if train else uncertainty_loss.eval()
    device = cfg["device"]; total_loss = 0.0
    all_labels, all_probs, all_days_gt, all_days_pred = [], [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            out = model(batch)
            loss, parts = compute_loss(out, batch, uncertainty_loss, cfg, device)
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(list(model.parameters()) + list(uncertainty_loss.parameters()), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item()
            probs = torch.softmax(out["cls_logits"], dim=1).detach().cpu().numpy()
            all_labels.extend(batch["label"].cpu().numpy().tolist())
            all_probs.extend(probs.tolist())
            all_days_gt.extend(batch["days_until"].cpu().numpy().tolist())
            all_days_pred.extend(out["reg_output"].squeeze(1).detach().cpu().numpy().tolist())
    all_probs_arr = np.array(all_probs)
    if train:
        all_preds = all_probs_arr.argmax(axis=1).tolist(); threshold_used = 0.5
    else:
        if threshold is None:
            raise ValueError("run_epoch(train=False) requires a pre-calibrated threshold.")
        threshold_used = threshold
        all_preds = (all_probs_arr[:, 1] >= threshold_used).astype(int).tolist()
    f1 = f1_score(all_labels, all_preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs_arr[:, 1])
    except ValueError:
        auc = float("nan")
    mae = mean_absolute_error(all_days_gt, all_days_pred)
    return {"f1": round(f1, 4), "auc": round(auc, 4), "mae": round(mae, 4),
            "loss": round(total_loss / max(len(loader), 1), 4), "threshold": threshold_used}


def get_cfg():
    return {"epochs": 30, "batch_size": 4, "lr": 1e-4, "lr_finetune": 1e-5, "weight_decay": 1e-4,
            "cls_class_weights": [1.0, 2.3], "device": DEVICE}


# Part A: Two-Level Bootstrap on the 7 Ablation Configs (val only)

## Data (val + train for calibration)

In [ ]:
cfg = get_cfg()
train_ds = DayLevelSequenceDataset(split="train")
val_ds = DayLevelSequenceDataset(split="val", shared_pixel_cache=train_ds._pixel_cache)

train_loader_calib = DataLoader(train_ds, batch_size=1, shuffle=False, collate_fn=day_sequence_collate, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=day_sequence_collate, num_workers=0)

VAL_LABELS = np.array([int(t["label"]) for t in val_ds.trajectories])
N_VAL = len(VAL_LABELS)
print(f"val: {N_VAL} trajectories, {VAL_LABELS.sum()} spoiled / {N_VAL - VAL_LABELS.sum()} not_spoiled")


## Checkpoint discovery -- however many seeds actually exist per config

In [ ]:
def discover_seeds(config_key, max_seed_search=30):
    found = []
    for seed in range(max_seed_search):
        ckpt_path = ABLATION_OUT / f"{config_key}_seed{seed}" / f"seed_{seed}" / "model_state.pt"
        if ckpt_path.exists():
            found.append(seed)
    return found


def build_model_for_config(config_key):
    return AblationModelCustomRGB(
        ABLATION_CONFIGS[config_key], d_model=64, dropout=0.5, cls_dropout=0.5,
        freeze_visual_backbone=True,
        rgb_encoder_factory=lambda: VisualEncoderToggle("convnext_tiny", True, use_cbam=True),
    ).to(cfg["device"])


SEEDS_FOUND = {}
for config_key in CONFIG_ORDER:
    seeds = discover_seeds(config_key)
    SEEDS_FOUND[config_key] = seeds
    print(f"  {CONFIG_LABELS[config_key]:<32} ({config_key}): {len(seeds)} seed(s) -> {seeds}")


## Collect per-seed val probabilities and per-seed F1 for every config

This single collection pass feeds both A.1 (ensemble-level) and A.2 (seed-level) below -- each seed's own val F1 (at its own calibrated threshold) is kept alongside the raw probabilities, since A.2 needs the 10 actual F1 numbers themselves, not anything derived from resampling trajectories.

In [ ]:
CONFIG_SEED_PROBS = {}       # config_key -> (n_seeds, n_val) array of P(spoiled)
CONFIG_SEED_THRESHOLDS = {}  # config_key -> list of per-seed thresholds
CONFIG_SEED_F1 = {}          # config_key -> list of per-seed val F1 (at that seed's own threshold)

for config_key in CONFIG_ORDER:
    seeds = SEEDS_FOUND[config_key]
    if not seeds:
        print(f"  [SKIP] {config_key}: no checkpoints found.")
        continue
    seed_probs, seed_thrs, seed_f1s = [], [], []
    for seed in seeds:
        try:
            ckpt_path = ABLATION_OUT / f"{config_key}_seed{seed}" / f"seed_{seed}" / "model_state.pt"
            model = build_model_for_config(config_key)
            model.load_state_dict(torch.load(ckpt_path, map_location=cfg["device"]))
            model.eval()

            val_probs, val_labels_check = collect_probs_labels(model, val_loader)
            assert list(val_labels_check) == list(VAL_LABELS), \
                f"val label order mismatch for {config_key} seed {seed}"
            seed_probs.append(val_probs[:, 1])

            train_probs, train_labels = collect_probs_labels(model, train_loader_calib)
            thr, _ = calibrate_threshold(train_probs[:, 1].tolist(), train_labels)
            seed_thrs.append(thr)

            seed_preds = (val_probs[:, 1] >= thr).astype(int)
            seed_f1 = f1_score(VAL_LABELS, seed_preds, average="binary", zero_division=0)
            seed_f1s.append(seed_f1)

            del model
        except Exception as e:
            tb = traceback.format_exc()
            print(f"  [FAIL] [{config_key}] seed {seed}: {type(e).__name__}: {e}\n{tb}")
            FAILURES.append({"config": config_key, "seed": seed, "stage": "collection",
                              "type": type(e).__name__, "message": str(e), "traceback": tb})

    if seed_probs:
        CONFIG_SEED_PROBS[config_key] = np.stack(seed_probs)
        CONFIG_SEED_THRESHOLDS[config_key] = seed_thrs
        CONFIG_SEED_F1[config_key] = seed_f1s
        print(f"  [{config_key}] collected {len(seed_probs)} seeds. "
              f"per-seed F1: {[round(f, 3) for f in seed_f1s]}")

print(f"\n{len(CONFIG_SEED_PROBS)} / {len(CONFIG_ORDER)} configs usable. "
      f"{len(FAILURES)} failure(s) so far.")


## A.1 -- Ensemble-level trajectory bootstrap

Averages probabilities across all discovered seeds (matching exactly how `05` computed its reported ensemble F1), thresholds with the median of the seeds' own calibrated thresholds, then bootstraps by resampling **trajectories** with replacement, 10,000 times, using the same resampled indices for every config within an iteration so the resulting draws stay paired. This quantifies: if a different validation set had been drawn, how much would ensemble F1 move?

In [ ]:
N_BOOTSTRAP = 10000
rng_ensemble = np.random.default_rng(0)

available_configs = list(CONFIG_SEED_PROBS.keys())
CONFIG_ENSEMBLE_PROBS = {c: CONFIG_SEED_PROBS[c].mean(axis=0) for c in available_configs}
CONFIG_ENSEMBLE_THRESHOLD = {c: float(np.median(CONFIG_SEED_THRESHOLDS[c])) for c in available_configs}
CONFIG_ENSEMBLE_OBSERVED_F1 = {
    c: f1_score(VAL_LABELS, (CONFIG_ENSEMBLE_PROBS[c] >= CONFIG_ENSEMBLE_THRESHOLD[c]).astype(int),
                average="binary", zero_division=0)
    for c in available_configs
}

BOOT_ENSEMBLE_F1 = {c: np.zeros(N_BOOTSTRAP) for c in available_configs}
for b in range(N_BOOTSTRAP):
    idx = rng_ensemble.integers(0, N_VAL, size=N_VAL)
    resampled_labels = VAL_LABELS[idx]
    for c in available_configs:
        resampled_probs = CONFIG_ENSEMBLE_PROBS[c][idx]
        resampled_preds = (resampled_probs >= CONFIG_ENSEMBLE_THRESHOLD[c]).astype(int)
        BOOT_ENSEMBLE_F1[c][b] = f1_score(resampled_labels, resampled_preds, average="binary", zero_division=0)
    if (b + 1) % 2500 == 0:
        print(f"  A.1: {b + 1}/{N_BOOTSTRAP} done")

print("A.1 (ensemble-level, trajectory resampling) complete.")
for c in available_configs:
    lo, hi = np.percentile(BOOT_ENSEMBLE_F1[c], [2.5, 97.5])
    print(f"  {CONFIG_LABELS[c]:<32} ensemble F1={CONFIG_ENSEMBLE_OBSERVED_F1[c]:.4f} "
          f"95% CI=[{lo:.4f}, {hi:.4f}]")


## A.2 -- Seed-level bootstrap

A genuinely different question from A.1: instead of resampling which trajectories count, this resamples **which seeds** count, drawing 10,000 samples of size N (with replacement) from each config's own N real single-seed F1 values, and takes the mean of each resample. This quantifies: if a different set of random seeds had been run, how much would the single-seed-mean F1 move? Kept explicitly separate from A.1 -- conflating the two would misrepresent which source of uncertainty is actually driving any instability in the reported numbers.

In [ ]:
rng_seed_level = np.random.default_rng(1)

CONFIG_SEED_MEAN_F1 = {c: float(np.mean(CONFIG_SEED_F1[c])) for c in available_configs}

BOOT_SEED_MEAN_F1 = {}
for c in available_configs:
    seed_f1_array = np.array(CONFIG_SEED_F1[c])
    n = len(seed_f1_array)
    resampled_means = np.zeros(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        draw = rng_seed_level.choice(seed_f1_array, size=n, replace=True)
        resampled_means[b] = draw.mean()
    BOOT_SEED_MEAN_F1[c] = resampled_means

print("A.2 (seed-level, resampling from the real per-seed F1 values) complete.")
for c in available_configs:
    lo, hi = np.percentile(BOOT_SEED_MEAN_F1[c], [2.5, 97.5])
    print(f"  {CONFIG_LABELS[c]:<32} single-seed-mean F1={CONFIG_SEED_MEAN_F1[c]:.4f} "
          f"95% CI=[{lo:.4f}, {hi:.4f}]  (n={len(CONFIG_SEED_F1[c])} seeds)")


## A.3 -- Paired delta vs. Full Model, under BOTH views

This is where the rank-inversion question raised in the notebook's opening cell actually gets answered with numbers. For every other config, two independent paired delta tests against Full Model: one under the ensemble view (A.1), one under the seed-level view (A.2). A config is explicitly flagged if the two views disagree on **significance** (CI excludes zero under one view but not the other) or on **direction** (the sign of the observed delta itself flips between views) -- either is a real rank-inversion, not a rounding difference.

In [ ]:
def paired_delta_table(boot_dict, observed_dict, metric_label):
    if "full_model" not in boot_dict:
        return None
    full_boot = boot_dict["full_model"]
    full_observed = observed_dict["full_model"]
    rows = []
    for c in available_configs:
        if c == "full_model":
            continue
        diff_boot = boot_dict[c] - full_boot
        observed_diff = observed_dict[c] - full_observed
        lo, hi = np.percentile(diff_boot, [2.5, 97.5])
        frac_le = float((diff_boot <= 0).mean()); frac_ge = float((diff_boot >= 0).mean())
        p = min(1.0, 2 * min(frac_le, frac_ge))
        significant = not (lo <= 0 <= hi)
        rows.append({
            "config": CONFIG_LABELS[c], "key": c, "view": metric_label,
            "observed_delta": round(observed_diff, 4),
            "ci_lower": round(float(lo), 4), "ci_upper": round(float(hi), 4),
            "p_value_approx": round(p, 4), "significant": significant,
        })
    return pd.DataFrame(rows)


ensemble_delta_df = paired_delta_table(BOOT_ENSEMBLE_F1, CONFIG_ENSEMBLE_OBSERVED_F1, "ensemble")
seedlevel_delta_df = paired_delta_table(BOOT_SEED_MEAN_F1, CONFIG_SEED_MEAN_F1, "seed_level")

print("Ensemble-level view (A.1) -- delta vs. Full Model:")
print(ensemble_delta_df.to_string(index=False))
print("\nSeed-level view (A.2) -- delta vs. Full Model:")
print(seedlevel_delta_df.to_string(index=False))


In [ ]:
merged = ensemble_delta_df.merge(seedlevel_delta_df, on=["config", "key"], suffixes=("_ensemble", "_seed_level"))
merged["direction_flip"] = np.sign(merged["observed_delta_ensemble"]) != np.sign(merged["observed_delta_seed_level"])
merged["significance_flip"] = merged["significant_ensemble"] != merged["significant_seed_level"]
merged["any_inversion"] = merged["direction_flip"] | merged["significance_flip"]

display_cols = ["config", "observed_delta_ensemble", "significant_ensemble",
                 "observed_delta_seed_level", "significant_seed_level",
                 "direction_flip", "significance_flip", "any_inversion"]
comparison_table = merged[display_cols].sort_values("any_inversion", ascending=False)
comparison_table.to_csv(OUT_DIR / "partA_two_view_comparison.csv", index=False)

print("=" * 100)
print("SIDE-BY-SIDE: ensemble view vs. seed-level view, per config")
print("=" * 100)
print(comparison_table.to_string(index=False))

inverted = comparison_table[comparison_table["any_inversion"]]
print(f"\n{'=' * 100}")
if inverted.empty:
    print("No config shows a direction flip or a significance flip between the two views. "
          "Whatever rank ordering appears in the raw ablation table is at least consistent "
          "in SIGN and SIGNIFICANCE between the ensemble and seed-level ways of measuring it, "
          "even if neither view reaches statistical significance on its own.")
else:
    print(f"{len(inverted)} config(s) show a real inversion between views:")
    for _, row in inverted.iterrows():
        flags = []
        if row["direction_flip"]:
            flags.append("DIRECTION FLIPS (better under one view, worse under the other)")
        if row["significance_flip"]:
            flags.append("SIGNIFICANCE FLIPS (distinguishable from Full Model under one view, not the other)")
        print(f"  {row['config']}: {'; '.join(flags)}")
        print(f"    ensemble delta={row['observed_delta_ensemble']:+.4f} (sig={row['significant_ensemble']}), "
              f"seed-level delta={row['observed_delta_seed_level']:+.4f} (sig={row['significant_seed_level']})")


# Part B: Leak-Safe 6-Fold CV Robustness Check (Full Model only)

## Leak-safe pool construction

**Not the same thing `13_kfold_cv.ipynb` did.** `13` built its pool with `DayLevelSequenceDataset(split=None)`, which pools every row regardless of split *before* day-level aggregation -- since aggregation merges same-day rows together, this does not just risk admitting whole test trajectories, it can merge individual train/val/test **days** into the same trajectory object. The pool here is built with an explicit `row_filter` restricting to `split in {train, val}` at the row level, before any aggregation happens.

In [ ]:
def train_val_only_filter(row):
    return row.get("split", "").strip().lower() in ("train", "val")


kfold_pool_ds = DayLevelSequenceDataset(split=None, row_filter=train_val_only_filter,
                                         shared_pixel_cache=train_ds._pixel_cache)
print(f"Leak-safe pool: {len(kfold_pool_ds)} trajectories "
      f"(restricted to train+val rows only, at the row level, before aggregation)")


## Programmatic leak assertion -- not a comment, an actual check

Every day-entry in every trajectory in the pool carries the `split` value of the exact manifest row it came from (threaded through unchanged from `TrimodalDataset._build_samples` in Part 1). This asserts directly against that real, per-row value, not against an assumption that the `row_filter` above worked correctly. A second sanity check confirms test rows do actually exist in the raw manifest at all -- otherwise the assertion below would trivially pass for the wrong reason (nothing to leak, not "successfully excluded what was there to exclude").

In [ ]:
# Sanity check: test rows actually exist in the raw manifest (so the assertion below
# is meaningful, not vacuously true).
with open(MANIFEST, newline="") as f:
    raw_rows = list(csv.DictReader(f))
raw_split_counts = defaultdict(int)
for r in raw_rows:
    raw_split_counts[r.get("split", "").strip().lower()] += 1
print(f"Raw manifest split counts: {dict(raw_split_counts)}")
assert raw_split_counts.get("test", 0) > 0, \
    "No 'test' rows found in the raw manifest at all -- the leak assertion below would be " \
    "vacuous. Check MANIFEST path and the 'split' column before trusting this notebook's " \
    "leak-safety claim."

# The actual leak check: no day-entry in the pool may have split == 'test'.
leaked_days = []
for traj in kfold_pool_ds.trajectories:
    for d in traj["days"]:
        if d["split"] == "test":
            leaked_days.append({"fruit": traj["fruit"], "label": traj["label"], "day_idx": d["day_idx"]})

assert len(leaked_days) == 0, (
    f"LEAK DETECTED: {len(leaked_days)} day-entries with split=='test' found in the "
    f"k-fold training pool. This must be fixed before any training starts. "
    f"Examples: {leaked_days[:5]}"
)

pool_split_counts = defaultdict(int)
for traj in kfold_pool_ds.trajectories:
    for d in traj["days"]:
        pool_split_counts[d["split"]] += 1
print(f"Pool split counts (should contain only train/val): {dict(pool_split_counts)}")
print(f"\n[PASS] No test-split day-entry found in the {len(kfold_pool_ds)}-trajectory "
      f"k-fold pool. Verified programmatically against the actual 'split' column, "
      f"not assumed from the row_filter alone.")


## 6-fold stratified split, matching the fold count used elsewhere in this project's threshold calibration

In [ ]:
N_FOLDS = 6
pool_labels = [int(t["label"]) for t in kfold_pool_ds.trajectories]
print(f"Pool: {len(pool_labels)} trajectories, {sum(pool_labels)} spoiled / "
      f"{len(pool_labels) - sum(pool_labels)} not_spoiled")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_indices = list(skf.split(np.zeros(len(pool_labels)), pool_labels))
for i, (tr_idx, va_idx) in enumerate(fold_indices):
    va_labels = [pool_labels[j] for j in va_idx]
    print(f"  Fold {i}: train={len(tr_idx)} val={len(va_idx)} (val spoiled={sum(va_labels)}/{len(va_labels)})")


## Train Full Model fresh, per fold

Cannot reuse existing checkpoints here, since fold composition changes what's actually in the training set each time. Same protocol as `04_fusion_model.ipynb`: 30 epochs, staged backbone unfreeze at epochs 6/8/12, differential learning rate for the visual backbones once unfrozen.

In [ ]:
D_MODEL, DROPOUT = 64, 0.5
KFOLD_OUT = OUT_DIR / "partB_kfold"
KFOLD_OUT.mkdir(parents=True, exist_ok=True)


class PoolSubsetDataset(Dataset):
    """Thin wrapper so OfflineAugmentedDayLevelSequenceDataset (which expects a
    .trajectories list) can be built on top of an index subset of kfold_pool_ds."""
    def __init__(self, base_ds, indices):
        self.base = base_ds
        self.indices = list(indices)
        self.trajectories = [base_ds.trajectories[i] for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[self.indices[i]]


def train_one_fold(fold_idx, train_idx, val_idx):
    fold_dir = KFOLD_OUT / f"fold_{fold_idx}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    set_seed(fold_idx)

    train_subset = PoolSubsetDataset(kfold_pool_ds, train_idx)
    val_subset = PoolSubsetDataset(kfold_pool_ds, val_idx)
    train_aug = OfflineAugmentedDayLevelSequenceDataset(train_subset, "flip_rotation", n_copies=1)

    train_loader_fold = DataLoader(train_aug, batch_size=cfg["batch_size"],
                                    sampler=build_weighted_sampler(train_aug),
                                    collate_fn=day_sequence_collate, num_workers=0)
    train_calib_loader_fold = DataLoader(train_subset, batch_size=cfg["batch_size"], shuffle=False,
                                          collate_fn=day_sequence_collate, num_workers=0)
    val_loader_fold = DataLoader(val_subset, batch_size=cfg["batch_size"], shuffle=False,
                                  collate_fn=day_sequence_collate, num_workers=0)

    model = TrimodalFusionModel(rgb_backbone_name="convnext_tiny", ir_backbone_name="efficientnet_b0",
                                 d_model=D_MODEL, dropout=DROPOUT, cls_dropout=DROPOUT,
                                 freeze_visual_backbone=True).to(cfg["device"])
    uncertainty_loss = UncertaintyWeightedLoss(n_tasks=3).to(cfg["device"])

    backbone_ids = {id(p) for p in list(model.rgb_encoder.backbone.parameters()) + list(model.ir_encoder.backbone.parameters())}
    other_params = [p for p in model.parameters() if id(p) not in backbone_ids]
    optimizer = AdamW([
        {"params": other_params, "lr": cfg["lr"]},
        {"params": list(model.rgb_encoder.backbone.parameters()) + list(model.ir_encoder.backbone.parameters()), "lr": cfg["lr_finetune"]},
        {"params": uncertainty_loss.parameters(), "lr": cfg["lr"]},
    ], weight_decay=cfg["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg["epochs"])

    print(f"{'-' * 90}\nFold {fold_idx}: train={len(train_idx)} val={len(val_idx)}\n{'-' * 90}")
    best_f1, best_state, best_metrics = -1.0, None, None
    history = []
    for epoch in range(1, cfg["epochs"] + 1):
        if epoch == 6:
            model.unfreeze_visual_last_n(1)
        elif epoch == 8:
            model.unfreeze_visual_last_n(2)
        elif epoch == 12:
            model.unfreeze_visual_full()

        train_m = run_epoch(model, uncertainty_loss, train_loader_fold, optimizer, cfg, train=True)
        thr = get_calibrated_threshold(model, train_calib_loader_fold)
        val_m = run_epoch(model, uncertainty_loss, val_loader_fold, optimizer, cfg, train=False, threshold=thr)
        scheduler.step()

        is_best = val_m["f1"] > best_f1
        if is_best:
            best_f1 = val_m["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = val_m

        history.append({"epoch": epoch, "train_f1": train_m["f1"], "val_f1": val_m["f1"],
                         "val_auc": val_m["auc"], "val_mae": val_m["mae"], "threshold": thr})
        print(f"  fold {fold_idx} | epoch {epoch:2d}/{cfg['epochs']} | train_F1={train_m['f1']:.3f} | "
              f"val_F1={val_m['f1']:.3f} val_AUC={val_m['auc']:.3f} val_MAE={val_m['mae']:.2f} "
              f"thr={thr:.2f}{'  *NEW BEST*' if is_best else ''}")

    torch.save(best_state, fold_dir / "model_state.pt")
    with open(fold_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2, default=str)
    print(f"  fold {fold_idx} DONE, best val F1={best_f1:.4f}")
    return best_metrics


In [ ]:
kfold_results = {}
KFOLD_FAILURES = []

for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    try:
        kfold_results[fold_idx] = train_one_fold(fold_idx, train_idx, val_idx)
    except Exception as e:
        tb = traceback.format_exc()
        print(f"  [FAIL] fold {fold_idx}: {type(e).__name__}: {e}\n{tb}")
        KFOLD_FAILURES.append({"fold": fold_idx, "type": type(e).__name__, "message": str(e), "traceback": tb})
        FAILURES.append({"stage": "kfold_training", "fold": fold_idx, "type": type(e).__name__,
                          "message": str(e), "traceback": tb})

with open(KFOLD_OUT / "kfold_results.json", "w") as f:
    json.dump(kfold_results, f, indent=2, default=str)
print(f"\n{len(kfold_results)} / {N_FOLDS} folds completed. {len(KFOLD_FAILURES)} failure(s).")


## Aggregate, and compare against three independent estimates for the same question

In [ ]:
if kfold_results:
    f1s = [m["f1"] for m in kfold_results.values()]
    aucs = [m["auc"] for m in kfold_results.values()]
    maes = [m["mae"] for m in kfold_results.values()]

    print("=" * 90)
    print(f"  K-FOLD CV -- FULL MODEL ({N_FOLDS} folds, leak-safe train+val pool)")
    print("=" * 90)
    print(f"  {'Fold':<8} {'F1':>8} {'AUC':>8} {'MAE':>8}")
    for i, m in kfold_results.items():
        print(f"  {i:<8} {m['f1']:>8.4f} {m['auc']:>8.4f} {m['mae']:>8.4f}")
    print("-" * 90)
    print(f"  {'Mean':<8} {np.mean(f1s):>8.4f} {np.mean(aucs):>8.4f} {np.mean(maes):>8.4f}")
    print(f"  {'Std':<8} {np.std(f1s):>8.4f} {np.std(aucs):>8.4f} {np.std(maes):>8.4f}")

    kfold_summary = {"f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s)),
                      "auc_mean": float(np.mean(aucs)), "auc_std": float(np.std(aucs)),
                      "mae_mean": float(np.mean(maes)), "mae_std": float(np.std(maes))}
    with open(KFOLD_OUT / "kfold_summary.json", "w") as f:
        json.dump(kfold_summary, f, indent=2)

    print(f"\n{'=' * 90}\n  THREE INDEPENDENT UNCERTAINTY ESTIMATES FOR FULL MODEL\n{'=' * 90}")
    print(f"  1. 04's single-split ensemble result (10-seed ensemble, one train/val/test split):")
    print(f"     see 04_fusion_model.ipynb's own reported val/test metrics directly.")
    print(f"  2. Part A's ensemble bootstrap CI (trajectory resampling, this notebook):")
    if "full_model" in CONFIG_ENSEMBLE_OBSERVED_F1:
        lo, hi = np.percentile(BOOT_ENSEMBLE_F1["full_model"], [2.5, 97.5])
        print(f"     ensemble F1 = {CONFIG_ENSEMBLE_OBSERVED_F1['full_model']:.4f}, "
              f"95% CI = [{lo:.4f}, {hi:.4f}]")
    else:
        print(f"     [unavailable -- see Part A / FAILURES]")
    print(f"  3. Part B's k-fold CV (this section):")
    print(f"     F1 = {np.mean(f1s):.4f} +/- {np.std(f1s):.4f} across {len(f1s)} folds")
    print(f"\n  Worth checking by eye whether these three broadly agree or point in different "
          f"directions -- they are not the same computation and are not guaranteed to match "
          f"exactly, but a large disagreement between them would itself be a finding.")
else:
    print("[SKIP] No folds completed successfully -- see KFOLD_FAILURES / FAILURES above.")


# Part C: One-Time Bootstrap CI on the Final Test-Set F1

**This is post-hoc reporting, not a decision-making step, and not an additional test touch in the sense that matters for this project's single-test-touch discipline.** The test evaluation itself already happened exactly once, in `04_fusion_model.ipynb`, using its 10-seed ensemble and its own k-fold-calibrated threshold. Nothing here reselects a checkpoint, retunes a threshold, or changes that already-final result -- this step only recomputes the deterministic forward pass of those exact, already-chosen checkpoints on test in order to get per-trajectory probabilities to bootstrap over, and reports a 95% CI around a number that cannot change as a result of running this cell.

In [ ]:
test_ds = DayLevelSequenceDataset(split="test", shared_pixel_cache=train_ds._pixel_cache)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=day_sequence_collate, num_workers=0)
TEST_LABELS = np.array([int(t["label"]) for t in test_ds.trajectories])
N_TEST = len(TEST_LABELS)
print(f"test: {N_TEST} trajectories, {TEST_LABELS.sum()} spoiled / {N_TEST - TEST_LABELS.sum()} not_spoiled")

trainval_ds = DayLevelSequenceDataset(split=None, row_filter=train_val_only_filter,
                                       shared_pixel_cache=train_ds._pixel_cache)
trainval_loader_calib = DataLoader(trainval_ds, batch_size=1, shuffle=False,
                                    collate_fn=day_sequence_collate, num_workers=0)
print(f"train+val (calibration only): {len(trainval_ds)} trajectories")


## Load the 10 already-trained Full Model seeds, recompute test probabilities

Threshold: median of each seed's own threshold calibrated on the combined train+val set. This is a documented simplification relative to `04`'s own fuller k-fold-over-train+val calibration procedure -- consistent with the same simplification already used elsewhere in this project's diagnostic notebooks, and stated here rather than presented as an exact reproduction of `04`'s number.

In [ ]:
full_model_seed_test_probs = []
full_model_seed_thresholds = []
PARTC_FAILURES = []

for seed in range(10):
    ckpt_path = FUSION_OUT / RUN_KEY / f"seed_{seed}" / "model_state.pt"
    if not ckpt_path.exists():
        print(f"  [SKIP] seed {seed}: checkpoint not found at {ckpt_path}")
        continue
    try:
        model = TrimodalFusionModel(rgb_backbone_name="convnext_tiny", ir_backbone_name="efficientnet_b0",
                                     d_model=D_MODEL, dropout=DROPOUT, cls_dropout=DROPOUT,
                                     freeze_visual_backbone=True).to(cfg["device"])
        model.load_state_dict(torch.load(ckpt_path, map_location=cfg["device"]))
        model.eval()

        test_probs, test_labels_check = collect_probs_labels(model, test_loader)
        assert list(test_labels_check) == list(TEST_LABELS), f"test label order mismatch for seed {seed}"
        full_model_seed_test_probs.append(test_probs[:, 1])

        trainval_probs, trainval_labels = collect_probs_labels(model, trainval_loader_calib)
        thr, _ = calibrate_threshold(trainval_probs[:, 1].tolist(), trainval_labels)
        full_model_seed_thresholds.append(thr)

        del model
        print(f"  seed {seed}: collected, threshold={thr:.3f}")
    except Exception as e:
        tb = traceback.format_exc()
        print(f"  [FAIL] seed {seed}: {type(e).__name__}: {e}\n{tb}")
        PARTC_FAILURES.append({"seed": seed, "type": type(e).__name__, "message": str(e), "traceback": tb})
        FAILURES.append({"stage": "partC_collection", "seed": seed, "type": type(e).__name__,
                          "message": str(e), "traceback": tb})

print(f"\n{len(full_model_seed_test_probs)} / 10 seeds collected for Part C.")


In [ ]:
if full_model_seed_test_probs:
    ensemble_test_probs = np.mean(full_model_seed_test_probs, axis=0)
    ensemble_test_threshold = float(np.median(full_model_seed_thresholds))
    ensemble_test_preds = (ensemble_test_probs >= ensemble_test_threshold).astype(int)
    observed_test_f1 = f1_score(TEST_LABELS, ensemble_test_preds, average="binary", zero_division=0)

    print(f"Recomputed test F1 = {observed_test_f1:.4f} at threshold={ensemble_test_threshold:.3f} "
          f"(compare against 04's own reported test F1 -- should match closely; a large "
          f"discrepancy would indicate the threshold-calibration simplification above matters "
          f"more than expected, worth a closer look if so)")
else:
    print("[SKIP] No Full Model test-seed checkpoints could be loaded -- Part C cannot proceed.")
    observed_test_f1 = None


## Bootstrap CI on the test F1

Same paired trajectory-resampling technique as Part A.1, applied once, to the fixed test predictions above.

In [ ]:
if observed_test_f1 is not None:
    rng_test = np.random.default_rng(2)
    boot_test_f1 = np.zeros(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        idx = rng_test.integers(0, N_TEST, size=N_TEST)
        resampled_labels = TEST_LABELS[idx]
        resampled_probs = ensemble_test_probs[idx]
        resampled_preds = (resampled_probs >= ensemble_test_threshold).astype(int)
        boot_test_f1[b] = f1_score(resampled_labels, resampled_preds, average="binary", zero_division=0)

    ci_lo, ci_hi = np.percentile(boot_test_f1, [2.5, 97.5])
    print("=" * 90)
    print("  POST-HOC 95% BOOTSTRAP CI ON THE ALREADY-FINAL TEST F1")
    print("=" * 90)
    print(f"  Observed test F1 (this recomputation): {observed_test_f1:.4f}")
    print(f"  Bootstrap mean: {boot_test_f1.mean():.4f}  |  std: {boot_test_f1.std():.4f}")
    print(f"  95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  (n={N_TEST} test trajectories -- as small a sample as val, so treat this "
          f"interval's width the same way: as a real, wide reflection of genuine "
          f"uncertainty, not a computational artifact.)")

    with open(OUT_DIR / "partC_test_bootstrap.json", "w") as f:
        json.dump({"observed_test_f1": float(observed_test_f1), "threshold": ensemble_test_threshold,
                    "ci_lower": float(ci_lo), "ci_upper": float(ci_hi),
                    "boot_mean": float(boot_test_f1.mean()), "boot_std": float(boot_test_f1.std()),
                    "n_test_trajectories": int(N_TEST)}, f, indent=2)
    print(f"\nSaved -> {OUT_DIR / 'partC_test_bootstrap.json'}")


## Closing summary: all three parts

In [ ]:
print("=" * 90)
print("SUMMARY")
print("=" * 90)

print("\nPART A (val, two-level bootstrap on 7 configs):")
if inverted.empty:
    print("  No config shows a direction or significance flip between the ensemble-level "
          "and seed-level views -- the rank-inversion question raised at the top of this "
          "notebook does not appear to be a real inversion, at least not one strong enough "
          "to show up as a sign or significance change under resampling.")
else:
    print(f"  {len(inverted)} config(s) show a real inversion between views -- see the Part A.3 "
          f"table above for exactly which, and in which direction.")

print("\nPART B (leak-safe 6-fold CV, Full Model only):")
if kfold_results:
    print(f"  F1 = {np.mean(f1s):.4f} +/- {np.std(f1s):.4f} across {len(f1s)} folds. "
          f"Compare directly against Part A's ensemble bootstrap CI and 04's own single-split "
          f"result above -- three independent estimates of the same underlying quantity.")
else:
    print("  No folds completed -- see KFOLD_FAILURES.")

print("\nPART C (post-hoc CI on the already-final test F1):")
if observed_test_f1 is not None:
    print(f"  Test F1 = {observed_test_f1:.4f}, 95% CI = [{ci_lo:.4f}, {ci_hi:.4f}]. "
          f"This interval's width is itself the headline finding here: read it before "
          f"treating the point estimate alone as precise.")
else:
    print("  Could not be computed -- see PARTC_FAILURES.")

print(f"\nTotal failures logged across all three parts: {len(FAILURES)}")
if FAILURES:
    print("See FAILURES for full type/message/traceback of each before treating any gap "
          "above as a deliberate exclusion rather than an error.")

print(f"\nAll outputs saved under: {OUT_DIR}")
print("\nThis notebook made no config-selection or architecture decision. It exists "
      "entirely to quantify how much confidence the results in 04 and 05 deserve.")
